# ⚡ Notebook 5: Performance, Atomicity & Probabilistic Structures

The first four notebooks covered the *what*. This one covers the *how-to-make-it-fast-and-correct*:

- **Pipelines** — kill network round-trips
- **Transactions (`MULTI`/`EXEC`) and `WATCH`** — atomic and optimistic-locking workflows
- **Lua scripting** — run multi-step logic inside Redis, atomically
- **Persistence (RDB vs AOF)** — what survives a restart, and at what cost
- **Bitmaps** — millions of booleans in a few KB
- **HyperLogLog** — count *unique* things in 12 KB regardless of how many

## Learning Objectives
- Use pipelines to make bulk operations 10–100× faster
- Understand the difference between Redis transactions and SQL transactions
- Use `WATCH` for optimistic concurrency control
- Write a small Lua script and understand why it's atomic
- Pick between RDB snapshots, AOF, or both
- Use Bitmaps for daily active users
- Use HyperLogLog for unique visitor counts


## 🛠️ Setup

```bash
cd 03-technologies/databases/redis
docker compose up -d
```

### Visualization
- **RedisInsight**: http://localhost:5540 — connect to `redis://localhost:6379`

### Kernel Selection
Select the `.venv` kernel in VS Code's kernel picker (top-right).
If it doesn't appear, reload: `Cmd+Shift+P` → "Reload Window".


In [ ]:
import redis
import time
import random

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

try:
    r.ping()
    print("✅ Connected to Redis")
    r.flushdb()
    print("🧹 Flushed database for a clean start")
except Exception as e:
    print(f"❌ Redis connection failed: {e}")
    print("   Run: cd 03-technologies/databases/redis && docker compose up -d")


## 1️⃣ Pipelines — destroy network round-trips

Each Redis command from your app is normally one network round-trip. At ~100 µs per round-trip on a fast LAN, **10,000 commands ≈ 1 second of pure network time**, even though Redis itself processed them in microseconds.

A **pipeline** sends many commands together in one socket write, then reads all the replies in one socket read. The commands aren't atomic (other clients can interleave), but they're dramatically faster.

> **Pipeline ≠ Transaction.** Pipelines are about *throughput*; transactions are about *atomicity*. We'll do transactions next.


In [ ]:
N = 5000

# Bad: one round-trip per SET
start = time.time()
for i in range(N):
    r.set(f"plain:{i}", i)
plain_ms = (time.time() - start) * 1000
print(f"Without pipeline: {N:>5} SETs in {plain_ms:7.1f} ms")

# Best: one batched round-trip for all SETs
start = time.time()
pipe = r.pipeline(transaction=False)  # transaction=False = pure pipeline (faster)
for i in range(N):
    pipe.set(f"piped:{i}", i)
pipe.execute()
piped_ms = (time.time() - start) * 1000
print(f"With pipeline   : {N:>5} SETs in {piped_ms:7.1f} ms")

print(f"\nSpeedup: {plain_ms / piped_ms:.1f}x")
print("\n[INSIGHT] Pipelines are the single biggest perf win when you do many commands in a row.")
print("          Use them for bulk loads, multi-key reads, fan-out writes, etc.")


## 2️⃣ Transactions (`MULTI` / `EXEC`) — and how they differ from SQL

A Redis transaction queues commands and runs them as one atomic block — **no other client can interleave between them**.

But there are two surprises for people coming from SQL:

| You might expect (SQL)     | Redis actually does                              |
| -------------------------- | ------------------------------------------------ |
| Rollback on error          | ❌ No rollback. Successful commands stay applied. |
| Read inside the transaction| ❌ Replies arrive only when `EXEC` runs.          |

So Redis transactions are best for **"do these N writes atomically"**, not for **"read X, decide, then write Y atomically"**. For that, we need **`WATCH`** (optimistic locking) — see the next cell.


In [ ]:
r.flushdb()
r.set("counter", 0)

# MULTI / EXEC: queue commands, send them as one atomic batch
pipe = r.pipeline(transaction=True)
pipe.incr("counter")
pipe.incr("counter")
pipe.incrby("counter", 10)
results = pipe.execute()
print(f"Results of each queued command: {results}")
print(f"Final counter value:            {r.get('counter')}")

# Demonstrate that a failed command does NOT undo earlier successful ones
print("\nDemonstrating no-rollback behaviour:")
r.set("name", "alice")
pipe = r.pipeline(transaction=True)
pipe.set("name", "bob")          # OK
pipe.incr("name")                # FAILS at runtime ("bob" is not an integer)
pipe.set("language", "redis")    # OK - still applied
try:
    pipe.execute()
except redis.exceptions.ResponseError as e:
    print(f"  EXEC reported an error: {e}")

print(f"  name     = {r.get('name')}     (still 'bob' - the failed INCR did NOT undo this)")
print(f"  language = {r.get('language')} (still applied after the failed command)")


### `WATCH` — optimistic locking for "read, decide, write" workflows

`WATCH key` tells Redis: *"if this key changes between now and my `EXEC`, **abort** my transaction (`EXEC` returns `None`)"*. The client typically retries in a loop.

This is how you implement safe **read-modify-write** patterns — the canonical example is "transfer N units from account A to account B, but only if A has enough".


In [ ]:
r.flushdb()
r.mset({"account:alice": 100, "account:bob": 50})

def transfer(src, dst, amount, max_retries=5):
    """Atomic transfer: only succeeds if the source balance hasn't changed mid-flight."""
    for attempt in range(max_retries):
        with r.pipeline() as pipe:
            try:
                pipe.watch(src)                                # start watching src
                src_balance = int(pipe.get(src) or 0)          # immediate read
                if src_balance < amount:
                    pipe.unwatch()
                    return False, f"insufficient funds (balance={src_balance})"

                pipe.multi()                                   # start the queued block
                pipe.decrby(src, amount)
                pipe.incrby(dst, amount)
                pipe.execute()                                 # commits IFF src didn't change
                return True, "ok"
            except redis.exceptions.WatchError:
                # Someone else modified src between WATCH and EXEC -> retry
                continue
    return False, "max retries exceeded"

ok, msg = transfer("account:alice", "account:bob", 30)
print(f"Transfer 30: ok={ok} ({msg})")
print(f"  alice = {r.get('account:alice')}, bob = {r.get('account:bob')}")

ok, msg = transfer("account:alice", "account:bob", 999)
print(f"Transfer 999: ok={ok} ({msg})")
print(f"  alice = {r.get('account:alice')}, bob = {r.get('account:bob')}")

print("\n[INSIGHT] WATCH = optimistic concurrency control. No locks held server-side;")
print("          the transaction simply aborts if the watched key was touched.")


## 3️⃣ Lua scripts — atomic, multi-step server-side logic

Sometimes a workflow is too dynamic for `MULTI`/`EXEC` (you need branches, loops, or to read a value and decide what to do next). Redis lets you ship a small **Lua script** — it runs **atomically** on the server (no other client gets in between), and you avoid round-trips.

We already used one in Notebook 3 (the safe-release lock). Here's a pattern you'll see all the time: a **token bucket** rate limiter, where the limiter logic *must* be atomic.


In [ ]:
r.flushdb()

# Token-bucket rate limiter implemented entirely server-side in Lua.
# KEYS[1] = bucket key
# ARGV[1] = capacity (max tokens)   ARGV[2] = refill_rate (tokens per second)
# ARGV[3] = now (unix seconds)      ARGV[4] = requested tokens
# Returns: { allowed (1/0), tokens_remaining, retry_after_seconds }
TOKEN_BUCKET_LUA = '''
local capacity   = tonumber(ARGV[1])
local rate       = tonumber(ARGV[2])
local now        = tonumber(ARGV[3])
local requested  = tonumber(ARGV[4])

local data = redis.call("HMGET", KEYS[1], "tokens", "ts")
local tokens = tonumber(data[1])
local ts     = tonumber(data[2])

if tokens == nil then
    tokens = capacity
    ts = now
end

-- Refill based on elapsed time
local elapsed = math.max(0, now - ts)
tokens = math.min(capacity, tokens + elapsed * rate)

local allowed = 0
local retry = 0
if tokens >= requested then
    tokens = tokens - requested
    allowed = 1
else
    retry = (requested - tokens) / rate
end

redis.call("HMSET", KEYS[1], "tokens", tokens, "ts", now)
redis.call("EXPIRE", KEYS[1], math.ceil(capacity / rate) * 2)

return { allowed, tokens, tostring(retry) }
'''

token_bucket = r.register_script(TOKEN_BUCKET_LUA)

# 5-token bucket that refills at 1 token/second
print("Bucket: capacity=5, refill=1 token/sec")
for i in range(8):
    allowed, remaining, retry = token_bucket(
        keys=["bucket:user_42"],
        args=[5, 1, time.time(), 1],
    )
    label = "allowed" if int(allowed) else f"DENIED (retry in {float(retry):.2f}s)"
    print(f"  request {i+1}: {label:35s} remaining={float(remaining):.2f}")

print("\n... waiting 3 seconds for the bucket to refill ...")
time.sleep(3)
allowed, remaining, retry = token_bucket(
    keys=["bucket:user_42"],
    args=[5, 1, time.time(), 1],
)
print(f"  after refill: allowed={int(allowed)}, remaining={float(remaining):.2f}")

print("\n[INSIGHT] Lua = atomic, single round-trip, server-side decisions.")
print("          Token bucket is the standard 'fair' rate limiter and nicely shows the pattern.")


## 4️⃣ Persistence — RDB vs AOF

Redis is in-memory, but it can persist to disk. There are two strategies (and you can combine them):

| Mode | What it writes                                | Recovery | Trade-off |
| ---- | --------------------------------------------- | -------- | --------- |
| **RDB** (snapshots) | A binary point-in-time dump of all data, every N seconds | Fast restart, small file | You can lose everything since the last snapshot |
| **AOF** (append-only file) | Every write command, appended to a log | Replays the log on startup | Larger file, can rewrite/compact periodically |
| **Both** | RDB snapshot + AOF since the snapshot | Best of both | Slightly more disk I/O |

The right choice depends on your durability needs:

- **Cache only?** Often no persistence at all (`save ""`). Fastest, and if the data is regenerable from the source of truth, you don't need it.
- **Sessions / counters?** RDB is usually plenty — losing a few seconds is fine.
- **Anything you can't recompute?** AOF with `appendfsync everysec` (the default), or `always` for max durability at the cost of throughput.

> Our `redis-master` container in `docker-compose.yml` uses `--appendonly yes` (AOF). The standalone `redis` container uses Redis 7 defaults (RDB, no AOF).

Let's just *inspect* the running config — don't change persistence settings on a real production server without thinking carefully.


In [ ]:
r2 = redis.Redis(host="localhost", port=6379, decode_responses=True)

print("--- Standalone Redis (port 6379) ---")
info = r2.info("persistence")
print(f"  RDB last save (unix ts):    {info.get('rdb_last_save_time')}")
print(f"  RDB changes since save:     {info.get('rdb_changes_since_last_save')}")
print(f"  RDB last bgsave status:     {info.get('rdb_last_bgsave_status')}")
print(f"  AOF enabled:                {info.get('aof_enabled')}")

# Trigger a synchronous snapshot (BGSAVE is the async, production-safe variant)
print("\nTriggering BGSAVE (asynchronous snapshot)...")
r2.bgsave()
time.sleep(0.5)
info = r2.info("persistence")
print(f"  RDB last save updated:      {info.get('rdb_last_save_time')}")

print("\n--- Master with AOF (port 6380) ---")
try:
    rm = redis.Redis(host="localhost", port=6380, decode_responses=True, socket_timeout=2)
    info = rm.info("persistence")
    print(f"  AOF enabled:                {info.get('aof_enabled')}")
    print(f"  AOF current size (bytes):   {info.get('aof_current_size')}")
    print(f"  AOF last rewrite status:    {info.get('aof_last_bgrewrite_status')}")
except Exception as e:
    print(f"  (master not reachable: {e})")

print("\n[INSIGHT] RDB = fast restart, possible data loss. AOF = better durability, larger file.")
print("          'appendfsync everysec' (the default) is the sweet spot for most apps.")


## 5️⃣ Bitmaps — millions of booleans, a few KB

A bitmap is just a Redis String, but you address its individual *bits*. That's a built-in, memory-tiny way to track a yes/no fact for billions of users.

Classic use case: **Daily Active Users**. One key per day, one bit per user ID. 10 million users = ~1.25 MB per day. Set/check/count are all `O(1)` to `O(N/8)`.


In [ ]:
r.flushdb()
today = "dau:2026-04-19"

# A few users were active today
for user_id in [1, 5, 17, 200, 999, 1_000_000]:
    r.setbit(today, user_id, 1)

print(f"User 5 active today?       {bool(r.getbit(today, 5))}")
print(f"User 6 active today?       {bool(r.getbit(today, 6))}")
print(f"User 1,000,000 active?     {bool(r.getbit(today, 1_000_000))}")

# BITCOUNT: total active users (all set bits)
print(f"\nTotal active users today:  {r.bitcount(today)}")

# Bytes used (one bit per user => 8x smaller than a Set of integers)
mem = r.memory_usage(today)
print(f"Bitmap memory:             {mem} bytes for IDs up to 1,000,000")

# Set logic between days: who was active BOTH yesterday and today?
yesterday = "dau:2026-04-18"
for user_id in [1, 17, 42, 1_000_000]:
    r.setbit(yesterday, user_id, 1)

r.bitop("AND", "dau:both", today, yesterday)
print(f"\nActive both days (BITOP AND -> BITCOUNT): {r.bitcount('dau:both')}")

print("\n[INSIGHT] Use Bitmaps when the universe of items has small, dense integer IDs.")
print("          Perfect for DAU, feature-flag-per-user, A/B test cohorts.")


## 6️⃣ HyperLogLog — count *unique* things at constant memory

When you don't need exact counts but you do need **unique cardinality** (e.g. "unique visitors per article"), HyperLogLog (HLL) is magical: it uses **~12 KB per key** to estimate distinct counts of *billions* of items, with about **0.81 % standard error**.

You can also `PFMERGE` multiple HLLs to get the union — perfect for rolling up daily counts to weekly.


In [ ]:
r.flushdb()

# Track unique visitors per page using HLL
for visitor in [f"user_{i}" for i in range(100_000)]:
    r.pfadd("uv:home", visitor)

# Add some duplicates - HLL doesn't double-count
for visitor in [f"user_{i}" for i in range(50_000)]:
    r.pfadd("uv:home", visitor)

estimate = r.pfcount("uv:home")
mem = r.memory_usage("uv:home")
print(f"True uniques fed in: 100,000")
print(f"PFCOUNT estimate:    {estimate:>7,}  (error {abs(estimate-100_000)/100_000*100:.2f}%)")
print(f"Memory used:         {mem:>7} bytes  (constant - doesn't grow with cardinality)")

# Union two HLLs to get combined uniques
for visitor in [f"user_{i}" for i in range(80_000, 180_000)]:
    r.pfadd("uv:about", visitor)

r.pfmerge("uv:total", "uv:home", "uv:about")
print(f"\nMerged uniques (home U about): {r.pfcount('uv:total'):,}")
print(f"True merged uniques:           180,000")

print("\n[INSIGHT] HLL trades exactness for memory. Use it for analytics-style 'how many uniques?'")
print("          When you need exact counts, use a Set (which costs O(N) memory).")


## 📋 Cheat Sheet

| Concern | Use | Why |
| ------- | --- | --- |
| Bulk operations are slow | **Pipeline** | Eliminates per-command round-trips |
| Two writes must succeed together | **MULTI/EXEC** | Atomic batch (no rollback though) |
| Read-modify-write across commands | **WATCH + MULTI/EXEC** | Optimistic locking, retry on conflict |
| Multi-step server-side logic | **Lua script** | Atomic + single round-trip |
| Cache only, can re-fetch | **No persistence** (`save ""`) | Max throughput |
| Mostly OK with a few seconds loss | **RDB** | Fast restart, small file |
| Cannot lose recent writes | **AOF** (`everysec`) | Replay log on startup |
| Per-user yes/no flag for millions | **Bitmap** | One bit per user, BITCOUNT/BITOP |
| Approximate unique counts | **HyperLogLog** | ~12 KB per key, ~0.81 % error |


## 🧹 Cleanup

In [ ]:
r.flushdb()
print("🧹 Cleaned up all keys from this notebook")


## 📚 Summary

### Key Takeaways

1. **Pipelines** can give 10–100× speedup on bulk operations — almost free win.
2. **Redis transactions** are atomic batches with **no rollback** — different from SQL.
3. **`WATCH`** is optimistic locking: abort + retry if a watched key changed.
4. **Lua scripts** run atomically server-side — perfect when you need branching logic.
5. **Persistence**: RDB for fast snapshots, AOF for durability — pick (or combine) based on your data's importance.
6. **Bitmaps** are wildly memory-efficient for boolean facts about dense integer IDs.
7. **HyperLogLog** gives you cardinality at constant memory — analytics without the pain.

### Where to go next

- Explore **Redis Streams** consumer groups in production (Notebook 2)
- Combine the **safe lock** (Notebook 3) with **Lua** (this notebook) for production-grade locking
- Learn **Redlock** — the multi-master variant of distributed locks
- Check **client-side caching** (Redis 6+) — `CLIENT TRACKING` for cache invalidation
